In [ ]:
import sys
from pathlib import Path

current_dir = Path.cwd().resolve()

# Find 'src' by walking up the directory tree
src_path = None
for parent in [current_dir] + list(current_dir.parents):
    candidate = parent / 'src'
    if candidate.exists() and candidate.is_dir():
        src_path = candidate
        break

if src_path:
    if str(src_path) not in sys.path:
        sys.path.append(str(src_path))
    print(f"Success: Linked to src at {src_path}")
    
    # Import
    try:
        from config import *
        print(f"Environment linked. Data dir: {DATA_DIR}")    
    except ImportError as e:
        print(f"CRITICAL: Found src but could not import config. {e}")
else:
    print("CRITICAL ERROR: Could not find 'src' directory in any parent folder.")
    print(f"Current search path: {current_dir}")

In [ ]:
CHUNK_SIZE = 1000

In [ ]:
def get_max_chunked(file_path):
    """
    Finds the maximum value in a large .npy file without loading it all into RAM.
    """
    data = np.load(file_path, mmap_mode='r') 
    global_max = 0.0
    
    # Iterate in chunks
    for i in range(0, len(data), CHUNK_SIZE):
        chunk = data[i : i + CHUNK_SIZE]
        chunk_max = np.max(np.abs(chunk))
        if chunk_max > global_max:
            global_max = chunk_max
            
    return global_max

In [ ]:
def verify_boundaries():
    """
    Check 1: Does the scaled data stay strictly within [-1, 1]?
    """
    print("--- Test 1: Boundary Verification ---")
    
    # We expect the TRAIN set to hit 1.0 exactly (since it contained the global max)
    train_path = get_unet_path(STAGE_SCALED, TRAIN, MIXED_DATASET)

    if train_path.exists():
        max_val_train = get_max_chunked(train_path)
        print(f"  Training Set Max: {max_val_train:.9f}")
        
        if np.isclose(max_val_train, 1.0):
            print("  PASS: Training set peaks exactly at 1.0.")
        else:
            print(f"  FAIL: Expected 1.0, got {max_val_train}")

    # Check Test Set
    test_path = get_unet_path(STAGE_SCALED, TEST, MIXED_DATASET)
    if test_path.exists():
        max_val_test = get_max_chunked(test_path)
        print(f"  Test Set Max:     {max_val_test:.9f}")
        
        if max_val_test <= 1.0:
            print("  PASS: Test set is strictly within [-1, 1] bounds.")
        else:
            print(f"  FAIL: Test set exceeds 1.0 ({max_val_test}).")

In [ ]:
def verify_reversibility(scaling_factor):
    """
    Check 2: Mathematical Reversibility
    - Logic: Can we recover the original hardware voltages? 
    - Formula: Original ≈ Scaled * Global_Factor
    """
    print("--- Test 2: Reversibility (Chunked) ---")
    
    mixed_path = get_unet_path(STAGE_MIXED, TRAIN, MIXED_DATASET)
    scaled_path = get_unet_path(STAGE_SCALED, TRAIN, MIXED_DATASET)
    
    if mixed_path.exists() and scaled_path.exists():
        # Open both files in read-only mode
        mixed_mmap = np.load(mixed_path, mmap_mode='r')
        scaled_mmap = np.load(scaled_path, mmap_mode='r')
        
        max_error = 0.0
        
        # Check just the first few chunks (checking the whole TB is slow and unnecessary)
        # We will check the first 5 chunks
        for i in range(0, min(len(mixed_mmap), CHUNK_SIZE*5), CHUNK_SIZE):
            m_chunk = mixed_mmap[i : i + CHUNK_SIZE]
            s_chunk = scaled_mmap[i : i + CHUNK_SIZE]
            
            reconstructed = s_chunk * scaling_factor
            chunk_error = np.max(np.abs(m_chunk - reconstructed))
            
            if chunk_error > max_error:
                max_error = chunk_error
                
        print(f"  Max Reconstruction Error (Sample): {max_error:.9e}")
        
        if max_error < 1e-5:
            print("  PASS: Scaling is fully reversible.")
        else:
            print("  FAIL: Significant data loss.")

In [ ]:
def verify_physics_preservation():
    """
    Check 3: Energy Ratio (SINR) Preservation
    - Logic: Global scaling must preserve the RELATIVE power between samples.
    - Test: If Sample A is 2x louder than Sample B in 'Mixed', it must be 2x louder in 'Scaled'.
    """
    print("--- Test 3: Physics Preservation (Lightweight) ---")
    
    # We only need 2 specific indices, so we don't need to load the whole file.
    # We can load just the specific indices we want.
    
    mixed_path = get_unet_path(STAGE_MIXED, VAL, MIXED_DATASET)
    scaled_path = get_unet_path(STAGE_SCALED, VAL, MIXED_DATASET)
    
    if mixed_path.exists() and scaled_path.exists():
        m_mmap = np.load(mixed_path, mmap_mode='r')
        s_mmap = np.load(scaled_path, mmap_mode='r')
        
        idx1, idx2 = 10, 20
        
        # Load ONLY the specific rows
        E_m1 = np.sum(np.abs(m_mmap[idx1])**2)
        E_m2 = np.sum(np.abs(m_mmap[idx2])**2)
        
        E_s1 = np.sum(np.abs(s_mmap[idx1])**2)
        E_s2 = np.sum(np.abs(s_mmap[idx2])**2)
        
        ratio_mixed = E_m1 / E_m2
        ratio_scaled = E_s1 / E_s2
        
        print(f"  Energy Ratio (Mixed):  {ratio_mixed:.6f}")
        print(f"  Energy Ratio (Scaled): {ratio_scaled:.6f}")
        
        if np.isclose(ratio_mixed, ratio_scaled, rtol=1e-5):
            print("  PASS: Relative Signal Power preserved.")
        else:
            print("  FAIL: Physics destroyed.")

In [ ]:
if SCALING_FACTORS_FILE.exists():
    with open(SCALING_FACTORS_FILE, 'r') as f:
        # Read and convert to float
        factor = float(f.read().strip())
    print(f"Loaded Scaling Factor: {factor}")
    
    verify_boundaries()
    verify_reversibility(factor)
    verify_physics_preservation()
    
    print("All System Checks Complete. Dataset is ready for U-Net Training.")
else:
    print(f"ERROR: Artifact not found at {SCALING_FACTORS_FILE}")